#POS (part-of-speech)
WEEK 5 PBA/ Irmasari Hafidz
Date: 24/ Sept/ 2024

Part-of-speech (POS) tagging is the process of assigning a part-of-speech tag (Noun, Verb, Adjective...) to each word in an input text.

Reference:
* https://github.com/jalajthanaki/POS-tag-workshop/blob/master/Introduction_to_POS.ipynb
* (ipynb) dari NLP Specialization on Coursera by deeplearning.ai
* More advanced POS tagger with Transformers (BERT) https://github.com/soutsios/pos-tagger-bert/blob/master/pos_tagger_bert.ipynb

##Data Scrapping
Data scrapping menggunakan BeautifulSoup dari 3 link berita, search keyword: CNN BPJS TBC

Link:

* link 1: https://news.detik.com/berita/d-7477855/program-jkn-bantu-mega-keluarga-dapat-akses-layanan-kesehatan-gratis

* link 2: https://www.cnnindonesia.com/ekonomi/20230828195349-92-991552/uang-bpjs-terkuras-rp10-t-imbas-penyakit-napas-termasuk-dampak-polusi

* link 3: https://www.cnnindonesia.com/ekonomi/20210715134614-83-668092/jalan-kesembuhan-habudin-lawan-tbc-bersama-jkn-kis



In [6]:
!pip install beautifulsoup4 requests


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Install necessary libraries (if not already installed)
!pip install beautifulsoup4 requests pandas

from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Load dataset and take URLs from column `url_resolved`
csv_candidates = [
    Path("out/berita_all_keywords.csv"),
    Path("../out/berita_all_keywords.csv"),
    Path("berita_all_keywords.csv"),
]

csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("berita_all_keywords.csv not found. Checked: out/, ../out/, current folder")

source_df = pd.read_csv(csv_path)
if "url_resolved" not in source_df.columns:
    raise KeyError("Column `url_resolved` not found in berita_all_keywords.csv")

urls = (
    source_df["url_resolved"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .tolist()
)

print(f"Loaded {len(urls)} URLs from {csv_path}")

# Function to scrape content from a URL
def scrape_content(url):
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract the title of the article
            title = soup.find('title').text if soup.find('title') else None

            # Extract the article's content
            paragraphs = soup.find_all('p')
            content = "\n".join([para.get_text(strip=True) for para in paragraphs])

            return {
                "url": url,
                "title": title,
                "content": content
            }
        else:
            return {
                "url": url,
                "title": None,
                "content": None,
                "error": f"Failed to fetch page, status code: {response.status_code}"
            }
    except Exception as e:
        return {
            "url": url,
            "title": None,
            "content": None,
            "error": str(e)
        }

# Scrape each URL and store the results in a list of dictionaries
data = []
for url in urls:
    result = scrape_content(url)
    data.append(result)

# Create a pandas DataFrame from the list of dictionaries
df = pd.DataFrame(data)

# Display the DataFrame
df.head()

# Save the DataFrame to a CSV file (optional)
df.to_csv('scraped_articles.csv', index=False)

Loaded 755 URLs from ..\out\berita_all_keywords.csv



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data Definition
Dataset adalah 3 link dengan kolom berisi: url, judul, konten.

In [8]:
df.head()

,url,title,content,error
0,https://tribratanews.polri.go.id/blog/nasional...,Wamendukbangga: Distribusi MBG untuk Ibu Hamil...,Tribratanews.polri.go.id– Jakarta. Wakil Mente...,NaN
1,https://www.tempo.co/kolom/mbg-dan-pertaruhan-...,Makan Gratis dan Pertaruhan Politik Anggaran |...,Iklan\nBerita Tempo Plus\nHandi Risza\nWakil R...,NaN
2,https://news.google.com/rss/articles/CBMirAFBV...,Google News,,NaN
3,https://video.tribunnews.com/news/925177/viral...,NaN,NaN,"Failed to fetch page, status code: 403"
4,https://jateng.jpnn.com/jateng-terkini/19807/m...,NaN,NaN,"Failed to fetch page, status code: 403"


In [9]:
# Install and download NLTK resources for POS tagging
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Frans\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Frans\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [10]:
from nltk import word_tokenize, pos_tag

In [11]:
# Importing packages and loading in the data set
import pandas as pd
from collections import defaultdict
import math
import numpy as np

In [12]:
# POS tagging preview for each article row
for index, row in df.iterrows():
    tokens = word_tokenize(str(row['content']))
    tags = pos_tag(tokens)
    print(f"Row {index}: {tags[:8]}\n")

Row 0: [('Tribratanews.polri.go.id', 'NNP'), ('–', 'NNP'), ('Jakarta', 'NNP'), ('.', '.'), ('Wakil', 'NNP'), ('Menteri', 'NNP'), ('(', '('), ('Wamen', 'NNP')]

Row 1: [('Iklan', 'NNP'), ('Berita', 'NNP'), ('Tempo', 'NNP'), ('Plus', 'NNP'), ('Handi', 'NNP'), ('Risza', 'NNP'), ('Wakil', 'NNP'), ('Rektor', 'NNP')]

Row 2: []

Row 3: [('nan', 'NN')]

Row 4: [('nan', 'NN')]

Row 5: [('nan', 'NN')]

Row 6: [('Pedagang', 'NNP'), ('buah', 'NN'), ('di', 'NN'), ('Pasar', 'NNP'), ('Gede', 'NNP'), ('Hardjonagoro', 'NNP'), ('Solo', 'NNP'), (',', ',')]

Row 7: []

Row 8: [('nan', 'NN')]

Row 9: [('Jauh', 'NNP'), ('sebelum', 'NN'), ('program', 'NN'), ('Makan', 'NNP'), ('Bergizi', 'NNP'), ('Gratis', 'NNP'), ('(', '('), ('MBG', 'NNP')]

Row 10: [('Persoalan', 'NNP'), ('program', 'NN'), ('Makan', 'NNP'), ('Bergizi', 'NNP'), ('Gratis', 'NNP'), ('(', '('), ('MBG', 'NNP'), (')', ')')]

Row 11: [('nan', 'NN')]

Row 12: [('nan', 'NN')]

Row 13: [('Apr', 'NNP'), ('6', 'CD'), (',', ','), ('2026', 'CD'), ('Home

## POS tag contoh 1st row data
Diambil dari df dari kolom [content] baris ke [1]

In [13]:
df_content1 = df['content'][1]
df_content1

'Iklan\nBerita Tempo Plus\nHandi Risza\nWakil Rektor Universitas Paramadina, lulus program doktor ekonomi Islam dari Universitas Islam Negeri Syarif Hidayatullah Jakarta\nMakan bergizi gratis bak pedang bermata dua. Bisa memicu kegagalan politik anggaran yang menjadi bumerang bagi penguasa.\n6 April 2026 | 07.15 WIB\nDengarkan artikel\nBagikan\nGabung Tempo Circle\nTAHUN 2026 menjadi momen pertaruhan politik yang berat bagi Presiden Prabowo Subianto. Di tengah melonjaknya harga minyak dunia akibat perang Iran dengan Israel yang dibantu Amerika Serikat, Prabowo akan mempertahankan proyekmakan bergizi gratisatau MBG.\nEdisi 5 April 2026\nMinyak Hilang Elpiji Tak Terbilang\nMinyak Hilang Elpiji Tak Terbilang\nPODCAST REKOMENDASI TEMPO\nIklan\nKebrutalan Polisi di Dogiyai\nSalah Kaprah DPR Mengurusi Pengadilan\nTebus Mahal Pemangkasan Dana Transfer ke Daerah\nSaatnya Menaikkan Harga BBM\nIklan\nTebus Dosa Mahkamah Konstitusi\nIlusi Pertumbuhan Lewat Kelas Berduit\nMemutus Rantai Korupsi Su

In [14]:
df_content1_tokens = word_tokenize(df_content1)
df_content1_tokens[:50]

['Iklan',
 'Berita',
 'Tempo',
 'Plus',
 'Handi',
 'Risza',
 'Wakil',
 'Rektor',
 'Universitas',
 'Paramadina',
 ',',
 'lulus',
 'program',
 'doktor',
 'ekonomi',
 'Islam',
 'dari',
 'Universitas',
 'Islam',
 'Negeri',
 'Syarif',
 'Hidayatullah',
 'Jakarta',
 'Makan',
 'bergizi',
 'gratis',
 'bak',
 'pedang',
 'bermata',
 'dua',
 '.',
 'Bisa',
 'memicu',
 'kegagalan',
 'politik',
 'anggaran',
 'yang',
 'menjadi',
 'bumerang',
 'bagi',
 'penguasa',
 '.',
 '6',
 'April',
 '2026',
 '|',
 '07.15',
 'WIB',
 'Dengarkan',
 'artikel']

In [15]:
# NLTK models already downloaded in setup cell above

In [16]:
df_content1_pos = pos_tag(df_content1_tokens)

print("{:<16}{}".format("Word", "POS Tag") + "\n" + "-" * 30)
for word, tag in df_content1_pos:
    print("{:<16}{:>2}".format(word, tag))

Word            POS Tag
------------------------------
Iklan           NNP
Berita          NNP
Tempo           NNP
Plus            NNP
Handi           NNP
Risza           NNP
Wakil           NNP
Rektor          NNP
Universitas     NNP
Paramadina      NNP
,                ,
lulus           JJ
program         NN
doktor          NN
ekonomi         VBP
Islam           NNP
dari            NN
Universitas     NNP
Islam           NNP
Negeri          NNP
Syarif          NNP
Hidayatullah    NNP
Jakarta         NNP
Makan           NNP
bergizi         NN
gratis          NN
bak             NN
pedang          NN
bermata         NN
dua             NN
.                .
Bisa            NNP
memicu          NN
kegagalan       NN
politik         VBP
anggaran        NN
yang            NN
menjadi         NN
bumerang        NN
bagi            NN
penguasa        NN
.                .
6               CD
April           NNP
2026            CD
|               NNP
07.15           CD
WIB             NNP
Dengarkan

## Latihan 1
Kerjakan POS tag untuk baris ke-2 dan ke-3.

In [17]:
import numpy as np
import pandas as pd

In [18]:
df_content1_pos

[('Iklan', 'NNP'),
 ('Berita', 'NNP'),
 ('Tempo', 'NNP'),
 ('Plus', 'NNP'),
 ('Handi', 'NNP'),
 ('Risza', 'NNP'),
 ('Wakil', 'NNP'),
 ('Rektor', 'NNP'),
 ('Universitas', 'NNP'),
 ('Paramadina', 'NNP'),
 (',', ','),
 ('lulus', 'JJ'),
 ('program', 'NN'),
 ('doktor', 'NN'),
 ('ekonomi', 'VBP'),
 ('Islam', 'NNP'),
 ('dari', 'NN'),
 ('Universitas', 'NNP'),
 ('Islam', 'NNP'),
 ('Negeri', 'NNP'),
 ('Syarif', 'NNP'),
 ('Hidayatullah', 'NNP'),
 ('Jakarta', 'NNP'),
 ('Makan', 'NNP'),
 ('bergizi', 'NN'),
 ('gratis', 'NN'),
 ('bak', 'NN'),
 ('pedang', 'NN'),
 ('bermata', 'NN'),
 ('dua', 'NN'),
 ('.', '.'),
 ('Bisa', 'NNP'),
 ('memicu', 'NN'),
 ('kegagalan', 'NN'),
 ('politik', 'VBP'),
 ('anggaran', 'NN'),
 ('yang', 'NN'),
 ('menjadi', 'NN'),
 ('bumerang', 'NN'),
 ('bagi', 'NN'),
 ('penguasa', 'NN'),
 ('.', '.'),
 ('6', 'CD'),
 ('April', 'NNP'),
 ('2026', 'CD'),
 ('|', 'NNP'),
 ('07.15', 'CD'),
 ('WIB', 'NNP'),
 ('Dengarkan', 'NNP'),
 ('artikel', 'NN'),
 ('Bagikan', 'NNP'),
 ('Gabung', 'NNP'),
 ('T

In [19]:
import pandas as pd

# In NLTK flow, df_content1_pos is already a list of (word, pos_tag) tuples.
# Create a list of dictionaries to store each word and its POS tag.
word_pos_list = [{"Word": word, "POS Tag": tag} for word, tag in df_content1_pos]

# Convert the list of dictionaries to a pandas DataFrame
df_word_pos = pd.DataFrame(word_pos_list)

# Display the DataFrame
print(df_word_pos)

# Optionally, save the DataFrame to a CSV file
df_word_pos.to_csv("word_pos_tags.csv", index=False)

            Word POS Tag
0          Iklan     NNP
1         Berita     NNP
2          Tempo     NNP
3           Plus     NNP
4          Handi     NNP
..           ...     ...
247        Iklan     NNP
248        Iklan     NNP
249        Iklan     NNP
250  TRUSTWORTHY     NNP
251         NEWS     NNP

[252 rows x 2 columns]


In [20]:
import pandas as pd
from collections import Counter

# In NLTK flow, df_content1_pos is already a list of (word, pos_tag) tuples.
# Create a list of dictionaries to store each word and its POS tag.
word_pos_list = [{"Word": word, "POS Tag": tag} for word, tag in df_content1_pos]

# Convert the list of dictionaries to a pandas DataFrame
df_word_pos = pd.DataFrame(word_pos_list)

# Count the occurrences of each POS tag
tag_counts = df_word_pos['POS Tag'].value_counts()

# Get the number of unique words for each POS tag
unique_tokens_per_tag = df_word_pos.groupby('POS Tag')['Word'].nunique()

# Combine the counts and unique tokens into a summary DataFrame
summary_df = pd.DataFrame({
    'Tag': tag_counts.index,
    'Count': tag_counts.values,
    'Unique Tokens': unique_tokens_per_tag
}).reset_index(drop=True)

# Sort by count in descending order
summary_df = summary_df.sort_values(by='Count', ascending=False)

# Display the summary DataFrame
print(summary_df)

# Optionally, save the summary DataFrame to a CSV file
summary_df.to_csv("pos_tags_summary.csv", index=False)

    Tag  Count  Unique Tokens
0   NNP    184              1
1    NN     36              2
2    CD      7              5
3    JJ      5              1
4     .      5              5
5   VBZ      4             32
6     ,      3            137
7   VBP      3              1
8    FW      3              1
9   NNS      1              3
10  VBD      1              3


In [21]:
# Calculate the sum of the 'Count' column
total_count = summary_df['Count'].sum()

# Display the result
print("Total Count:", total_count)

Total Count: 252
